# Customer Churn Prediction

A beginner-friendly end-to-end Machine Learning project that predicts whether a telecom customer is likely to leave the company.

**Target:** `Churn`
- `Yes` = customer left
- `No` = customer stayed

This notebook covers dataset understanding, cleaning, EDA, preprocessing, train/test split, scaling, model training, evaluation, comparison, and prediction.

## 1. Import libraries

We use Pandas/NumPy for data handling, Matplotlib for visualization, and Scikit-learn for preprocessing, models, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

pd.set_option('display.max_columns', None)

## 2. Load the dataset

In [ ]:
df = pd.read_csv('../data/customer_churn.csv')
df.head()

## 3. Understand the dataset

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### Important columns
- **CustomerID:** unique customer identifier
- **Gender:** customer gender
- **SeniorCitizen:** whether the customer is a senior citizen
- **Partner / Dependents:** family information
- **Tenure:** months the customer has stayed
- **InternetService:** internet service type
- **Contract:** contract type
- **PaymentMethod:** payment method
- **MonthlyCharges:** monthly bill
- **TotalCharges:** total amount paid
- **SupportCalls:** number of support calls
- **Churn:** target variable

## 4. Data cleaning checks

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# CustomerID is only an identifier and does not help the model learn churn behavior.
df_model = df.drop(columns=['CustomerID']).copy()

# Convert target: No = 0, Yes = 1
df_model['Churn'] = df_model['Churn'].map({'No': 0, 'Yes': 1})

df_model.head()

## 5. Exploratory Data Analysis (EDA)

EDA helps us understand patterns before training a model.

In [ ]:
churn_counts = df['Churn'].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(churn_counts.index, churn_counts.values)
plt.title('Customer Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.show()

print(churn_counts)

In [ ]:
plt.figure(figsize=(7, 4))
for churn_value in ['No', 'Yes']:
    subset = df[df['Churn'] == churn_value]
    plt.hist(subset['Tenure'], bins=8, alpha=0.6, label=churn_value)

plt.title('Tenure Distribution by Churn')
plt.xlabel('Tenure (months)')
plt.ylabel('Number of Customers')
plt.legend(title='Churn')
plt.show()

In [ ]:
contract_churn = pd.crosstab(df['Contract'], df['Churn'])
contract_churn.plot(kind='bar', figsize=(8, 4))
plt.title('Churn by Contract Type')
plt.xlabel('Contract')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.show()

contract_churn

In [ ]:
support_churn = df.groupby('Churn')['SupportCalls'].mean()
support_churn.plot(kind='bar', figsize=(6, 4))
plt.title('Average Support Calls by Churn')
plt.xlabel('Churn')
plt.ylabel('Average Support Calls')
plt.xticks(rotation=0)
plt.show()

support_churn

### EDA observation

In this small learning dataset, churn is commonly associated with shorter tenure, month-to-month contracts, higher support-call activity, and higher monthly charges.

## 6. Feature encoding

Machine Learning models require numerical input. `pd.get_dummies()` converts categorical text columns into 0/1 columns.

In [ ]:
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

print("Encoded shape:", X_encoded.shape)
X_encoded.head()

## 7. Train-test split

We split **before scaling**. This prevents data leakage.

- Training data: used to learn the model
- Testing data: used only to evaluate unseen data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 8. Feature scaling

We scale only numerical continuous/count features.

- `fit_transform()` on training data learns the mean/std and scales the training data.
- `transform()` on testing data uses the **same** mean/std.
- We do **not** fit the scaler on the test data.

In [ ]:
scale_columns = ['Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportCalls']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()

X_train_scaled[scale_columns] = scaler.fit_transform(X_train[scale_columns])
X_test_scaled[scale_columns] = scaler.transform(X_test[scale_columns])

X_train_scaled.head()

## 9. Train Logistic Regression

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

## 10. Evaluate Logistic Regression

In [ ]:
log_accuracy = accuracy_score(y_test, log_pred)
log_precision = precision_score(y_test, log_pred, zero_division=0)
log_recall = recall_score(y_test, log_pred, zero_division=0)
log_f1 = f1_score(y_test, log_pred, zero_division=0)

print("Accuracy :", round(log_accuracy, 3))
print("Precision:", round(log_precision, 3))
print("Recall   :", round(log_recall, 3))
print("F1 Score :", round(log_f1, 3))
print("\nClassification Report:\n")
print(classification_report(y_test, log_pred, target_names=['No Churn', 'Churn'], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, log_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn']).plot()
plt.title('Logistic Regression - Confusion Matrix')
plt.show()

### Confusion matrix meaning
- **True Negative (TN):** predicted No Churn and customer actually stayed
- **False Positive (FP):** predicted Churn but customer actually stayed
- **False Negative (FN):** predicted No Churn but customer actually left
- **True Positive (TP):** predicted Churn and customer actually left

For churn problems, **Recall** is important because a false negative means we failed to identify a customer who was going to leave.

## 11. Compare with Decision Tree

Decision Trees do not require scaling, so we train it on the unscaled encoded data.

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_pred)
tree_precision = precision_score(y_test, tree_pred, zero_division=0)
tree_recall = recall_score(y_test, tree_pred, zero_division=0)
tree_f1 = f1_score(y_test, tree_pred, zero_division=0)

model_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [log_accuracy, tree_accuracy],
    'Precision': [log_precision, tree_precision],
    'Recall': [log_recall, tree_recall],
    'F1 Score': [log_f1, tree_f1]
})

model_comparison.round(3)

## 12. Predict churn for one new customer

The new customer's values must use the same training columns. We encode the row, align columns, scale the numerical columns, then predict.

In [ ]:
new_customer = pd.DataFrame([{
    'Gender': 'Male',
    'SeniorCitizen': 'No',
    'Partner': 'No',
    'Dependents': 'No',
    'Tenure': 6,
    'InternetService': 'Fiber optic',
    'Contract': 'Month-to-month',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 95.0,
    'TotalCharges': 570.0,
    'SupportCalls': 5
}])

new_encoded = pd.get_dummies(new_customer, drop_first=True, dtype=int)
new_encoded = new_encoded.reindex(columns=X_encoded.columns, fill_value=0)

new_scaled = new_encoded.copy()
new_scaled[scale_columns] = scaler.transform(new_encoded[scale_columns])

prediction = log_model.predict(new_scaled)[0]
probability = log_model.predict_proba(new_scaled)[0][1]

print("Prediction:", "Churn" if prediction == 1 else "No Churn")
print("Churn probability:", round(probability * 100, 2), "%")

## 13. Final conclusion

This project demonstrates a complete beginner Machine Learning workflow:

1. Understand the business problem and target
2. Load and inspect data
3. Check missing values and duplicates
4. Remove an unnecessary ID column
5. Perform EDA
6. Encode categorical variables
7. Split into training and testing data
8. Fit scaling only on training data
9. Train Logistic Regression
10. Evaluate with Accuracy, Precision, Recall, F1 Score, and Confusion Matrix
11. Compare with a Decision Tree
12. Predict churn for a new customer

### Limitation
The included dataset has only **40 synthetic rows**, so the metrics are for learning and demonstration only. A production model needs a much larger real-world dataset.